> 在线讲义：[第 51 章 · Prefill–Decode 分离与资源配比：A100 做输入，H20 做生成](https://wikieden.github.io/ai-infra-from-constraints/lessons/51-pd-split.html) · 本书《从约束推导 AI Infra》

# 第 51 章 · Prefill–Decode 分离与资源配比：A100 做输入，H20 做生成

配套讲义：仓库里 `ai-infra/notes/lessons/51-pd-split.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



A100 / H20 的阶段能力按峰值 50% 从 Roofline 推出；例 9.1、9.2 与 §9.2.4 的配比，取自 AII 第 9 章 §9.2。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KiB, MiB, GiB = 2**10, 2**20, 2**30
K_GQA = 2 * 36 * 8 * 128 * 2            # Qwen3-8B 每 token 144 KiB
K_MLA = 61 * (512 + 64) * 2             # DeepSeek-V3 紧凑 MLA 每 token 70,272 B
D_W = 15_136_811_008                    # Qwen3-8B 一次 decode 读的权重字节
PREFILL_FLOP = 133.6e12                 # 8192 token prefill 的矩阵运算
DECODE_FLOP = 0.65e12                   # batch 32 一步 decode
CARDS = {"A100": (156e12, 1020e9), "H20": (74e12, 2048e9)}  # 有效算力、带宽（峰值 50%）

**函数 `v_kv`**：V_KV = 2 n L h_kv d_h b（MLA 时是 n L (d_c + d_r) b）。

In [ ]:
def v_kv(L: int, per_token: int = K_GQA) -> int:
    """V_KV = 2 n L h_kv d_h b（MLA 时是 n L (d_c + d_r) b）。"""
    return L * per_token

**函数 `handoff_ms`**：小工具。

In [ ]:
def handoff_ms(nbytes: float, bw: float = 25e9, alpha: float = 5e-6) -> float:
    return (nbytes / bw + alpha) * 1e3

**函数 `decode_step`**：一步 decode（秒）：max(计算, 读权重 + 读 KV)。

In [ ]:
def decode_step(card: str, batch: int = 32, ctx: int = 8704) -> float:
    """一步 decode（秒）：max(计算, 读权重 + 读 KV)。"""
    C, B = CARDS[card]
    return max(DECODE_FLOP / C, (D_W + batch * ctx * K_GQA) / B)

**函数 `stage_gpu_s`**：(d_P, d_D)：每请求在该卡上的 P、D GPU 秒。

In [ ]:
def stage_gpu_s(card: str, new_tokens: int = 8192, G: int = 1025, batch: int = 32, ctx: int | None = None):
    """(d_P, d_D)：每请求在该卡上的 P、D GPU 秒。"""
    C, _ = CARDS[card]
    d_p = PREFILL_FLOP * new_tokens / 8192 / C
    ctx = ctx if ctx is not None else 8192 + (G - 1) // 2
    d_d = (G - 1) * decode_step(card, batch, ctx) / batch
    return d_p, d_d

**函数 `pd_rate`**：μ_PD = min(μ_P, μ_D, B_net / V_KV)。d[card] = (d_P, d_D)。

In [ ]:
def pd_rate(p_cards: dict, d_cards: dict, d: dict, link: float = 25e9, state: float = 8192 * K_GQA):
    """μ_PD = min(μ_P, μ_D, B_net / V_KV)。d[card] = (d_P, d_D)。"""
    mp = sum(n / d[c][0] for c, n in p_cards.items())
    md = sum(n / d[c][1] for c, n in d_cards.items())
    return min(mp, md, link / state), mp, md

**函数 `colocated`**：小工具。

In [ ]:
def colocated(cards: dict, d: dict) -> float:
    return sum(n / sum(d[c]) for c, n in cards.items())

**函数 `piggyback_tokens`**：一步 decode 里空闲算力能捎带多少个 prefill token。

In [ ]:
def piggyback_tokens(card: str, batch: int = 32, ctx: int = 8704) -> float:
    """一步 decode 里空闲算力能捎带多少个 prefill token。"""
    C, _ = CARDS[card]
    idle = decode_step(card, batch, ctx) - DECODE_FLOP / C
    return idle * C / (PREFILL_FLOP / 8192)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：交接多少字节、多久")
for name, k in (("Qwen3-8B GQA", K_GQA), ("DeepSeek-V3 MLA", K_MLA)):
    v = v_kv(8192, k)
    print(f"  {name}：每 token {k / KiB:.1f} KiB，8K {v / MiB:.0f} MiB；25 GB/s {handoff_ms(v):.1f} ms，50 GB/s {handoff_ms(v, 50e9):.1f} ms")
v = v_kv(8192)
print(f"  双缓冲两端共 {2 * v / GiB:.2f} GiB；每秒 8 / 16 次交接要 {8 * v / 1e9:.1f} / {16 * v / 1e9:.1f} GB/s")

In [ ]:
print("\n第二步：两种卡的阶段能力")
d = {c: stage_gpu_s(c) for c in CARDS}
for c, (dp, dd) in d.items():
    print(f"  {c}：prefill {dp:.3f} s（{8192 / dp:.0f} token/s），decode 一步 {decode_step(c) * 1e3:.1f} ms（{32 / decode_step(c):.0f} 次/s），每请求 D {dd:.3f} GPU 秒")
print(f"  batch 32 生成到最后 KV {32 * 9216 * K_GQA / 1e9:.1f} GB")

In [ ]:
print("\n第三步：同构八张 A100")
print(f"  共置 {colocated({'A100': 8}, d):.2f} 请求/s；3P/5D → {pd_rate({'A100': 3}, {'A100': 5}, d)[0]:.2f}")
pb = piggyback_tokens("A100")
print(f"  分块捎带：每步空闲可做 {pb:.0f} 个 prefill token，8192 分 {-(-8192 // int(pb))} 块，首 token 约 {-(-8192 // int(pb)) * decode_step('A100'):.2f} s；理想上限 {8 / d['A100'][1]:.2f} 请求/s")

In [ ]:
print("\n第四步：异构 4×A100 + 4×H20")
print(f"  共置 {colocated({'A100': 4, 'H20': 4}, d):.2f} 请求/s")
mu, mp, md = pd_rate({"A100": 4}, {"H20": 4}, d)
print(f"  A100 做 P、H20 做 D：P {mp:.2f}、D {md:.2f}、链路 {25e9 / v:.1f} → {mu:.2f} 请求/s")
print(f"  角色对调：P 只剩 {pd_rate({'H20': 4}, {'A100': 4}, d)[1]:.2f}")
print(f"  MLA 状态链路项 {25e9 / v_kv(8192, K_MLA):.1f}；链路成为瓶颈的门槛 GQA {md * v / 1e9:.1f}、MLA {md * v_kv(8192, K_MLA) / 1e9:.1f} GB/s")
h = piggyback_tokens("H20")
left = 8192 - 32 * h
rate = 4 / d["A100"][1] + 4 / (d["H20"][1] + left / 8192 * d["H20"][0])
print(f"  H20 每步只能捎带 {h:.0f} 个；八卡理想捎带共置约 {rate:.2f} 请求/s（原文 4.17）")
grid = [[pd_rate({"A100": a, "H20": b}, {"A100": 4 - a, "H20": 4 - b}, d)[0] for b in range(5)] for a in range(5)]
best = max((grid[a][b], a, b) for a in range(5) for b in range(5))
print(f"  25 种配比里最好：{best[1]} 张 A100 + {best[2]} 张 H20 做 P → {best[0]:.2f}")

In [ ]:
print("\n第五步：请求变了，配比跟着变")
dh = {"A100": (0.2378, d["A100"][1]), "H20": (0.2378 * 156 / 74, d["H20"][1])}
print(f"  命中 6144 前缀（A100 prefill 0.238 s）：2 张 A100 做 P → {pd_rate({'A100': 2}, {'A100': 2, 'H20': 4}, dh)[0]:.2f}")
ds = {c: stage_gpu_s(c, G=129) for c in CARDS}
print(f"  输出 129：H20 D 能力 {1 / ds['H20'][1]:.2f}/卡；4A100+3H20 做 P → {pd_rate({'A100': 4, 'H20': 3}, {'H20': 1}, ds)[0]:.2f}，共置 {colocated({'A100': 4, 'H20': 4}, ds):.2f}")

## 3. 代码里才看得到的

1. decode 一步读的权重用 15,136,811,008 B（第 45 章同一个数），KV 按平均上下文 8704 算；换成 9216 会把 D 能力低估约 5%。
2. A100 分块捎带按代码算每步 487 个 token、理想上限 4.54（原文写 490 和 4.50）；H20 捎带后剩下的 5482 个 token 再按普通 prefill 算，得到 4.19（原文 4.17）。
3. 3P/5D 的 2.84 与原文 2.83 的差别来自 1.763 vs 1.764 的取整。

## 4. 自己改一改

- 遍历 `pd_rate` 的 25 种配比，画出自己的图 51-2。
- 把链路改成 4 GB/s，看最优配比还是不是 4A100 / 4H20。
- 给 `stage_gpu_s` 换一张"算力 400、带宽 3000"的假想卡，看它更适合哪边。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 交接班只交笔记，笔记大小 2nLhdb，传的时候两边都留着；
> 拉砖看发动机，送快递看路宽；
> 同一批厨师分组不省工时，先试顺手捎带；
> 会开车的开车、会搬货的搬货，两边一样快才不浪费；客人一变就重排。